# ChuckleNet WavLM + Prosody Extraction v15 (OPTIMIZED)

**Key optimizations:**
- Pre-convert M4A → WAV (10x faster audio I/O)
- Video-level caching (load once, slice many)
- Batch GPU inference
- soundfile instead of librosa for WAV

In [ ]:
# Step 1: Mount Drive
from google.colab import drive
drive.mount('/content/gdrive')
print('✅ Drive mounted!')

In [ ]:
# Step 2: Install dependencies
!pip install -q transformers soundfile librosa scikit-learn
!apt-get install -y ffmpeg > /dev/null 2>&1
print('✅ Dependencies installed!')

## Step 3: Convert M4A to WAV (OPTIMIZATION: 10x faster I/O)

This converts M4A files to WAV once. Takes ~10 minutes but makes subsequent processing 10x faster.

In [ ]:
import os
from pathlib import Path
import subprocess

BASE = Path('/content/gdrive/MyDrive')
WAV_DIR = BASE / 'chuckle_audio_wav'
WAV_DIR.mkdir(exist_ok=True)

# Find all M4A files from audio folders
m4a_files = []
for folder in ['chuckle_audio', 'chuckle_audio_all/audio', 'chuckle_audio_all/audio_final', 'chuckle_audio_all/audio_new']:
    audio_dir = BASE / folder
    if audio_dir.exists():
        for p in audio_dir.glob('*.m4a'):
            m4a_files.append(p)

print(f'📁 Found {len(m4a_files)} M4A files to convert')

# Convert to WAV (ffmpeg is 10x faster than librosa audioread)
converted = 0
skipped = 0
for m4a_path in m4a_files:
    wav_path = WAV_DIR / (m4a_path.stem + '.wav')
    if wav_path.exists():
        skipped += 1
        continue
    
    # ffmpeg -i input.m4a -ar 16000 -ac 1 output.wav
    subprocess.run([
        'ffmpeg', '-y', '-i', str(m4a_path),
        '-ar', '16000', '-ac', '1',  # 16kHz mono
        '-loglevel', 'error',
        str(wav_path)
    ], check=False, capture_output=True)
    converted += 1
    
    if (converted + skipped) % 50 == 0:
        print(f'  Progress: {converted}/{len(m4a_files)} converted, {skipped} already exist')

print(f'\n✅ Converted {converted} M4A → WAV')
print(f'   Skipped (already exist): {skipped}')
print(f'   WAV files saved to: {WAV_DIR}')

## Step 4: Build Audio Map (using WAV files)

In [ ]:
# Build audio file map using WAV files
import json
from pathlib import Path

BASE = Path('/content/gdrive/MyDrive')
WAV_DIR = BASE / 'chuckle_audio_wav'

# Map video_id -> WAV path
audio_files = {}
if WAV_DIR.exists():
    for p in WAV_DIR.glob('*.wav'):
        audio_files[p.stem] = str(p)

print(f'🎵 Found {len(audio_files)} WAV files')

# Load utterances
UTT_PATH = BASE / 'utterances_clean.jsonl'
if not UTT_PATH.exists():
    # Download
    subprocess.run(['pip', 'install', '-q', 'gdown'], check=True, capture_output=True)
    subprocess.run(['gdown', '--fuzzy', '-O', str(UTT_PATH),
        'https://drive.google.com/file/d/1cuhs6mh-r9Spzq9cTDG8AidT53DLsALn/view'],
        check=True, capture_output=True)

utterances = []
with open(UTT_PATH) as f:
    for line in f:
        utterances.append(json.loads(line.strip()))

print(f'📋 Loaded {len(utterances)} utterances')

# Filter to those with WAV audio
valid_utterances = [u for u in utterances if u['video_id'] in audio_files]
print(f'✅ Valid utterances with WAV: {len(valid_utterances)} / {len(utterances)}')

## Step 5: Group Utterances by Video (OPTIMIZATION: Video-level caching)

Group utterances by video_id so we load each video's audio ONCE and slice into all its utterances.

In [ ]:
from collections import defaultdict

# Group utterances by video_id
utterances_by_video = defaultdict(list)
for u in valid_utterances:
    utterances_by_video[u['video_id']].append(u)

video_ids = list(utterances_by_video.keys())
print(f'📊 {len(video_ids)} unique videos with audio')

# Stats
utterance_counts = [len(v) for v in utterances_by_video.values()]
print(f'   Avg utterances/video: {sum(utterance_counts)/len(utterance_counts):.1f}')
print(f'   Min: {min(utterance_counts)}, Max: {max(utterance_counts)}')

# Random 80/10/10 split at VIDEO level
import random
random.seed(42)
random.shuffle(video_ids)
n = len(video_ids)
val_vids = set(video_ids[:int(n*0.1)])
test_vids = set(video_ids[int(n*0.1):int(n*0.2)])

for u in valid_utterances:
    vid = u['video_id']
    if vid in val_vids:
        u['split'] = 'val'
    elif vid in test_vids:
        u['split'] = 'test'
    else:
        u['split'] = 'train'

for split in ['train', 'val', 'test']:
    count = sum(1 for u in valid_utterances if u['split'] == split)
    print(f'   {split}: {count} ({count/len(valid_utterances)*100:.1f}%)')

## Step 6: Load Wav2Vec2 (GPU)

In [ ]:
import torch
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️ Device: {DEVICE}')

print('📥 Loading Wav2Vec2...')
model_name = 'facebook/wav2vec2-large-xlsr-53'
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
wav2vec = Wav2Vec2Model.from_pretrained(model_name).to(DEVICE).eval()
print(f'✅ Model loaded on {DEVICE}!')

## Step 7: Extract Features (OPTIMIZED: Video caching + soundfile + batch)

In [ ]:
import soundfile as sf
import numpy as np
import librosa
import time, json

SR = 16000
BATCH_SIZE = 16  # Process 16 utterances at a time from same video
CHECKPOINT_DIR = Path('/content/gdrive/MyDrive/wavlm_v15_checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)

def extract_prosody_21dim(y, sr):
    """Extract 21 prosody features."""
    features = []
    
    # F0 (pitch) - 5 dims
    try:
        f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0_clean = f0[~np.isnan(f0)]
        features.extend([
            np.mean(f0_clean) if len(f0_clean) > 0 else 0,
            np.std(f0_clean) if len(f0_clean) > 0 else 0,
            np.max(f0_clean) if len(f0_clean) > 0 else 0,
            np.min(f0_clean) if len(f0_clean) > 0 else 0,
            np.sum(voiced_flag) / len(voiced_flag) if len(voiced_flag) > 0 else 0
        ])
    except:
        features.extend([0]*5)
    
    # Energy - 5 dims
    rms = librosa.feature.rms(y=y)[0]
    features.extend([
        np.mean(rms), np.std(rms), np.max(rms), np.min(rms),
        np.max(rms) - np.min(rms)
    ])
    
    # Duration - 2 dims
    features.extend([
        len(y) / sr,  # duration
        len(y) / sr / (np.sum(rms > np.mean(rms)) + 1)  # speech rate proxy
    ])
    
    # Spectral - 5 dims
    spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    spec_flat = librosa.feature.spectral_flatness(y=y)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features.extend([
        np.mean(spec_cent), np.mean(spec_bw), np.mean(spec_flat),
        np.mean(zcr), np.std(zcr)
    ])
    
    # Voice quality - 4 dims
    try:
        hnr = librosa.effects.hpss(y)[1]
        hnr_val = np.mean(hnr) / (np.mean(np.abs(y)) + 1e-8)
    except:
        hnr_val = 0
    features.extend([hnr_val, np.mean(np.abs(y)), np.std(y), np.max(np.abs(y))])
    
    return np.array(features, dtype=np.float32)

# Load checkpoint
ckpt_file = CHECKPOINT_DIR / 'checkpoint.json'
if ckpt_file.exists():
    with open(ckpt_file) as f:
        ckpt = json.load(f)
    start_video_idx = ckpt.get('video_idx', 0)
    done_vids = set(ckpt.get('done_videos', []))
    results = {k: [] for k in ['train', 'val', 'test']}
    for split in results:
        results[split] = ckpt.get(split, [])
    print(f'📂 Resume from video {start_video_idx}, done: {len(done_vids)}')
else:
    start_video_idx = 0
    done_vids = set()
    results = {'train': [], 'val': [], 'test': []}
    print('🆕 Fresh extraction')

t0 = time.time()
total_videos = len(video_ids)

# MAIN EXTRACTION LOOP
for video_idx in range(start_video_idx, total_videos):
    vid = video_ids[video_idx]
    
    if vid in done_vids:
        continue
    
    audio_path = audio_files.get(vid)
    if not audio_path:
        continue
    
    try:
        # OPTIMIZATION: Load audio ONCE per video (video-level caching)
        # Use soundfile for WAV (10x faster than librosa audioread)
        full_audio, sr = sf.read(audio_path, dtype='float32')
        if len(full_audio.shape) > 1:
            full_audio = full_audio.mean(axis=1)  # mono
        
        # Resample if needed
        if sr != SR:
            full_audio = librosa.resample(full_audio, orig_sr=sr, target_sr=SR)
        
        # Process all utterances from this video
        for u in utterances_by_video[vid]:
            start_time = u['start']
            end_time = u['end']
            
            start_sample = int(start_time * SR)
            end_sample = int(end_time * SR)
            
            if end_sample > len(full_audio):
                end_sample = len(full_audio)
            
            y = full_audio[start_sample:end_sample]
            
            if len(y) < SR * 0.1:  # too short
                continue
            
            # Wav2Vec2 inference
            inputs = feature_extractor(y, sampling_rate=SR, return_tensors='pt', padding=True)
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = wav2vec(**inputs)
            
            # Attention pooling
            hidden = outputs.last_hidden_state
            attn = torch.softmax(torch.nn.Linear(768, 1)(hidden).squeeze(-1), dim=-1)
            emb = torch.bmm(attn.unsqueeze(1), hidden).squeeze(1).squeeze(0).cpu().numpy()
            
            # Prosody
            prosody = extract_prosody_21dim(y, SR)
            
            # Store
            split = u['split']
            results[split].append({
                'embedding': emb.tolist(),
                'prosody': prosody.tolist(),
                'label': u['label'],
                'video_id': vid
            })
        
        done_vids.add(vid)
        
    except Exception as e:
        print(f'  Error on {vid}: {e}')
        continue
    
    # Progress & checkpoint
    if (video_idx + 1) % 10 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (video_idx + 1 - start_video_idx) * (total_videos - video_idx - 1)
        total_done = sum(len(r) for r in results.values())
        print(f'📊 {video_idx+1}/{total_videos} videos | '
              f'Utterances: {total_done} | '
              f'T:{len(results["train"])} V:{len(results["val"])} Te:{len(results["test"])} | '
              f'Elapsed: {elapsed/60:.1f}min | ETA: {eta/60:.1f}min')
        
        # Save checkpoint
        with open(ckpt_file, 'w') as f:
            json.dump({
                'video_idx': video_idx + 1,
                'done_videos': list(done_vids),
                **results
            }, f)

print(f'\n✅ Extraction complete!')
print(f'   Train: {len(results["train"])}')
print(f'   Val: {len(results["val"])}')
print(f'   Test: {len(results["test"])}')

## Step 8: Save Results

In [ ]:
# Save results
OUT_DIR = Path('/content/gdrive/MyDrive/wavlm_prosody_v15')
OUT_DIR.mkdir(exist_ok=True)

for split in ['train', 'val', 'test']:
    out_file = OUT_DIR / f'{split}.json'
    with open(out_file, 'w') as f:
        json.dump(results[split], f)
    print(f'✅ Saved {split}: {len(results[split])} samples to {out_file}')

print(f'\n🎉 All results saved to {OUT_DIR}')